<a href="https://colab.research.google.com/github/polreig/StartUp_DecoAI/blob/main/3_Busqueda_Productos.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Instalación

In [ ]:
!pip install -q -U google-genai

## Importar y conectar API

In [ ]:
from google import genai
from google.colab import userdata
import urllib.parse
from IPython.display import display, HTML

print("Conectando con Gemini...")
GOOGLE_API_KEY = userdata.get('clave_API_gemini')
gemini_client = genai.Client(api_key=GOOGLE_API_KEY)
print("¡Conectado!")

## Función búsqueda de links

In [ ]:
def generar_links_compra(texto_analisis):
    print("🛒 Extrayendo muebles clave y generando links...")

    # 1. Le pedimos a Gemini que extraiga 3 muebles
    prompt_extraccion = f"""
    Lee este análisis de diseño de interiores:
    '{texto_analisis}'

    Extrae los 3 muebles o elementos decorativos más importantes que se necesitan comprar para lograr este estilo.
    Responde ÚNICAMENTE con los nombres de los 3 elementos separados por comas.
    Ejemplo: sofá de cuero negro, lámpara de techo industrial, mesa de centro de metal
    """

    respuesta_muebles = gemini_client.models.generate_content(
        model='gemini-2.5-flash',
        contents=[prompt_extraccion]
    )

    # 2. Limpiamos la respuesta
    texto_muebles = respuesta_muebles.text.strip()
    lista_muebles = [mueble.strip() for mueble in texto_muebles.split(",")]

    # 3. Generamos el HTML con los links
    html_links = "<div style='background-color: #f9f9f9; padding: 15px; border-radius: 8px;'>"
    html_links += "<h3 style='margin-top: 0;'>🛍️ Lista de la Compra Recomendada:</h3><ul>"

    for mueble in lista_muebles:
        if not mueble: continue

        busqueda_codificada = urllib.parse.quote(mueble)
        link_ikea = f"https://www.ikea.com/es/es/search/?q={busqueda_codificada}"
        link_amazon = f"https://www.amazon.es/s?k={busqueda_codificada}"

        html_links += f"""
        <li style='margin-bottom: 10px;'>
            <b>{mueble.capitalize()}</b>:
            <a href='{link_ikea}' target='_blank' style='color: #0051ba; font-weight: bold; text-decoration: none;'>[ IKEA ]</a> |
            <a href='{link_amazon}' target='_blank' style='color: #ff9900; font-weight: bold; text-decoration: none;'>[ Amazon ]</a>
        </li>
        """

    html_links += "</ul></div>"
    return html_links

## Prueba

In [ ]:
# Simulamos lo que habría respondido el modelo de visión en el Cuaderno 1
texto_simulado = """
1.  **Estilo Actual:** La habitación actual presenta un estilo nórdico y minimalista, caracterizado por una paleta de colores neutros y materiales naturales, creando un ambiente sereno pero con poca vivacidad.

2.  **Recomendación de Estilo:** Te recomiendo un estilo **Boho-Chic Contemporáneo**, que te permitirá incorporar color y texturas orgánicas manteniendo la base de tus muebles actuales y añadiendo una sensación de calidez y personalidad relajada.
"""

resultado_html = generar_links_compra(texto_simulado)
display(HTML(resultado_html))